# SatCloudRestore — Colab GPU workflow
This notebook performs environment checks, pilot training, resumable quick training, staged evaluation, and Gradio launch. It contains no precomputed metrics. Replace `REPO_URL` before running.

In [ ]:
import platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime before continuing')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

In [ ]:
from pathlib import Path
REPO_URL = 'https://github.com/YOUR_ACCOUNT/SatCloudRestore.git'  # Replace with the real repository URL
PROJECT_DIR = Path('/content/SatCloudRestore')
if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
%cd /content/SatCloudRestore

In [ ]:
# Colab already provides a CUDA-enabled torch build; editable install keeps it when compatible.
%pip install -q -e . pytest

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from datetime import datetime
DRIVE_ROOT = Path('/content/drive/MyDrive/SatCloudRestore')
RUN_DATE = datetime.now().strftime('%Y%m%d')
PILOT_NAME = f'pilot_gpu_{RUN_DATE}'
QUICK_NAME = f'quick_{RUN_DATE}'
PILOT_CKPT = DRIVE_ROOT / 'checkpoints' / PILOT_NAME
QUICK_CKPT = DRIVE_ROOT / 'checkpoints' / QUICK_NAME
PILOT_RESULTS = DRIVE_ROOT / 'results' / PILOT_NAME
QUICK_RESULTS = DRIVE_ROOT / 'results' / QUICK_NAME
print('Drive root:', DRIVE_ROOT)

## Pre-training verification

In [ ]:
!python -m compileall satcloudrestore scripts app.py
!pytest -q

## Data and cloud demo

In [ ]:
!python scripts/prepare_data.py --config configs/quick.yaml
!python scripts/generate_demo.py --mode clouds --config configs/quick.yaml --output {DRIVE_ROOT / 'cloud_demo.png'}

## Pilot training (500 optimizer steps on T4/8–12 GB VRAM)

In [ ]:
!python scripts/train.py --config configs/pilot_gpu.yaml --device cuda --experiment-name {PILOT_NAME} --output-dir {PILOT_CKPT} --max-steps 500 --batch-size 16 --gradient-accumulation 4 --num-workers 2

If CUDA runs out of memory, start a new experiment directory and use `--batch-size 8 --gradient-accumulation 8` to retain effective batch size 64.

In [ ]:
!python scripts/evaluate.py --config configs/pilot_gpu.yaml --checkpoint {PILOT_CKPT / 'best.pt'} --device cuda --output-dir {PILOT_RESULTS / 'pilot_eval'} --max-images 40 --ddim-steps 25 --samples 1

## Formal quick training

In [ ]:
!python scripts/train.py --config configs/quick.yaml --device cuda --experiment-name {QUICK_NAME} --output-dir {QUICK_CKPT} --batch-size 32 --gradient-accumulation 2 --num-workers 2

## Resume after a Colab interruption (run instead of starting a new experiment)

In [ ]:
!python scripts/train.py --config configs/quick.yaml --device cuda --experiment-name {QUICK_NAME} --output-dir {QUICK_CKPT} --resume {QUICK_CKPT / 'latest.pt'} --batch-size 32 --gradient-accumulation 2 --num-workers 2

## Post-training verification

In [ ]:
!python -m compileall satcloudrestore scripts app.py
!pytest -q

## A. Full 1,000-image primary evaluation (`samples=1`)

In [ ]:
!python scripts/evaluate.py --config configs/quick.yaml --checkpoint {QUICK_CKPT / 'best.pt'} --device cuda --output-dir {QUICK_RESULTS / 'primary_1000'} --max-images 1000 --ddim-steps 25 --samples 1

## B. Selected 32-image uncertainty evaluation (`samples=8`)

In [ ]:
import json
manifest = json.loads(Path('data/splits/manifest.json').read_text())
test_rows = [row for row in manifest if row['split'] == 'test']
SELECTED_IDS = [row['image_id'] for row in test_rows[:32]]
SELECTED_ID_ARGS = ' '.join(SELECTED_IDS)
print('Selected images:', len(SELECTED_IDS))

In [ ]:
!python scripts/evaluate.py --config configs/quick.yaml --checkpoint {QUICK_CKPT / 'best.pt'} --device cuda --output-dir {QUICK_RESULTS / 'uncertainty_32'} --image-ids {SELECTED_ID_ARGS} --ddim-steps 25 --samples 8 --save-samples

## Gradio demo

In [ ]:
!python app.py --checkpoint {QUICK_CKPT / 'best.pt'} --config configs/quick.yaml --device cuda --share